In [1]:
import numpy as np
from collections import defaultdict, Counter
from gensim.models import Word2Vec, FastText
import string

# Parameters
min_count = 1
embedding_dim = 100
window = 5
learning_rate = 0.001
epochs = 10
x_max = 100
alpha = 0.75

# Define common sentences
common_sentences = [
    "The teacher likes to teach the class.",
    "The teacher is teaching the lesson today.",
    "The teacher teaches math and science.",
    "The teaching profession is noble.",
    "The teacher teaches the students.",
    "The teaching is happening in the classroom.",
    "The teacher is teaching computation.",
    "The computer can compute complex problems.",
    "The computer is computing the data.",
    "The computation is done by the computer.",
    "The computed results are displayed.",
    "The computational power is increasing.",
    "The computational science is advancing.",
    "The computer computed the answer.",
    "The natural world is full of wonders.",
    "The naturally beautiful landscape.",
    "The natural way to do it is naturally.",
    "The quick fox jumps over the lazy dog.",
    "The quick response is needed.",
    "The bird flies quickly.",
    "The quick car goes quickly."
]

# Define rare sentences
rare_sentences = [
    "The concept is unteachable.",
    "The recomputation was required.",
    "The unnaturalness was disturbing."
]

# Corpus
corpus = common_sentences * 10 + rare_sentences

# Clean words
trans = str.maketrans('', '', string.punctuation)
processed_corpus = [[word.translate(trans) for word in sentence.lower().split() if word.translate(trans)] for sentence in corpus]

# For GloVe, use the string corpus
# For gensim, use processed_corpus

# Build vocab for GloVe
word_count = Counter()
for sentence in processed_corpus:
    word_count.update(sentence)

vocab = [word for word in word_count if word_count[word] >= min_count]

# Build co-occurrence
co_occurrence = defaultdict(float)
for sentence in corpus:
    words = sentence.lower().split()
    words = [word.translate(trans) for word in words if word.translate(trans)]
    for i in range(len(words)):
        word = words[i]
        if word in vocab:
            for j in range(max(0, i - window), min(len(words), i + window + 1)):
                if i != j and words[j] in vocab:
                    distance = abs(i - j)
                    if distance > 0:
                        co_occurrence[(word, words[j])] += 1.0 / distance

# Initialize embeddings and biases with small values
word_embeddings = {word: np.random.randn(embedding_dim) * 0.01 for word in vocab}
bias = {word: 0.0 for word in vocab}

# Train GloVe
for epoch in range(epochs):
    total_loss = 0
    for (word_i, word_j), X in co_occurrence.items():
        if X > 0:
            f = min((X / x_max) ** alpha, 1.0)
            dot = np.dot(word_embeddings[word_i], word_embeddings[word_j])
            diff = dot + bias[word_i] + bias[word_j] - np.log(X)
            total_loss += f * (diff ** 2)
            grad = f * diff
            grad_wi = grad * word_embeddings[word_j]
            word_embeddings[word_i] -= learning_rate * grad_wi
            bias[word_i] -= learning_rate * grad
            grad_wj = grad * word_embeddings[word_i]
            word_embeddings[word_j] -= learning_rate * grad_wj
            bias[word_j] -= learning_rate * grad
    print(f"GloVe Epoch: {epoch+1}, Loss: {total_loss}")

# Train Word2Vec
word2vec = Word2Vec(processed_corpus, vector_size=embedding_dim, window=window, min_count=min_count, epochs=epochs)

# Train FastText
fasttext = FastText(processed_corpus, vector_size=embedding_dim, window=window, min_count=min_count, epochs=epochs)

# Functions for GloVe
def cosine_sim(embeddings, w1, w2):
    if w1 not in embeddings or w2 not in embeddings:
        return "N/A"
    v1 = embeddings[w1]
    v2 = embeddings[w2]
    return np.dot(v1, v2) / (np.linalg.norm(v1) * np.linalg.norm(v2))

def most_similar(embeddings, word, topn=5):
    if word not in embeddings:
        return "N/A"
    v = embeddings[word]
    similarities = {}
    for other in vocab:
        if other != word:
            sim = np.dot(v, embeddings[other]) / (np.linalg.norm(v) * np.linalg.norm( embeddings[other] ))
            similarities[other] = sim
    sorted_sim = sorted(similarities, key=similarities.get, reverse=True)
    return sorted_sim[:topn]

def most_similar_analogy(embeddings, positive, negative, topn=1):
    v = np.zeros(embedding_dim)
    inputs = positive + negative
    for word in positive:
        if word in embeddings:
            v += embeddings[word]
    for word in negative:
        if word in embeddings:
            v -= embeddings[word]
    similarities = {}
    for word in vocab:
        if word not in inputs:
            sim = np.dot(v, embeddings[word]) / (np.linalg.norm(v) * np.linalg.norm( embeddings[word] )) if np.linalg.norm(v) > 0 else 0
            similarities[word] = sim
    sorted_sim = sorted(similarities, key=similarities.get, reverse=True)
    return sorted_sim[:topn]

# Experiment 1: OOV Behavior
oov_words = ['teachable', 'unteacher', 'supercomputer', 'miscomputation', 'unnaturally']
print("\nExperiment 1: OOV Behavior")
print("GloVe:")
for word in oov_words:
    print(f"{word}: {'Yes' if word in vocab else 'No'}")

print("Word2Vec:")
for word in oov_words:
    print(f"{word}: {'Yes' if word in word2vec.wv else 'No'}")

print("FastText:")
for word in oov_words:
    try:
        fasttext.wv[word]
        print(f"{word}: Yes")
    except KeyError:
        print(f"{word}: No")

# Experiment 2: Rare Words
pairs = [('unteachable', 'teacher'), ('recomputation', 'computation'), ('unnaturalness', 'natural')]
print("\nExperiment 2: Rare Words")
print("GloVe:")
for p1, p2 in pairs:
    print(f"({p1}, {p2}): {cosine_sim(word_embeddings, p1, p2)}")

print("Word2Vec:")
for p1, p2 in pairs:
    print(f"({p1}, {p2}): {word2vec.wv.similarity(p1, p2)}" if p1 in word2vec.wv and p2 in word2vec.wv else f"({p1}, {p2}): N/A")

print("FastText:")
for p1, p2 in pairs:
    print(f"({p1}, {p2}): {fasttext.wv.similarity(p1, p2)}")

# Experiment 3: Morphological Relationships
words = ['teaching', 'computation']
print("\nExperiment 3: Morphological Relationships")
print("GloVe:")
for word in words:
    print(f"Top 5 similar to {word}: {most_similar(word_embeddings, word, 5)}")

print("Word2Vec:")
for word in words:
    print(f"Top 5 similar to {word}: {word2vec.wv.most_similar(word, topn=5)}")

print("FastText:")
for word in words:
    print(f"Top 5 similar to {word}: {fasttext.wv.most_similar(word, topn=5)}")

# Experiment 4: Morphological Analogies
analogies = [
    ('teacher', 'teaching', 'computer'),
    ('natural', 'naturally', 'quick')
]
print("\nExperiment 4: Morphological Analogies")
print("GloVe:")
for a, b, c in analogies:
    if all(x in vocab for x in [a, b, c]):
        predicted = most_similar_analogy(word_embeddings, [b, c], [a])[0]
        print(f"{a} : {b} = {c} : {predicted}")
    else:
        print(f"{a} : {b} = {c} : N/A")

print("Word2Vec:")
for a, b, c in analogies:
    if all(x in word2vec.wv for x in [a, b, c]):
        predicted = word2vec.wv.most_similar(positive=[b, c], negative=[a], topn=1)[0][0]
        print(f"{a} : {b} = {c} : {predicted}")
    else:
        print(f"{a} : {b} = {c} : N/A")

print("FastText:")
for a, b, c in analogies:
    predicted = fasttext.wv.most_similar(positive=[b, c], negative=[a], topn=1)[0][0]
    print(f"{a} : {b} = {c} : {predicted}")

GloVe Epoch: 1, Loss: 338.1209466629803
GloVe Epoch: 2, Loss: 327.99764949970296
GloVe Epoch: 3, Loss: 318.3121606013078
GloVe Epoch: 4, Loss: 309.0442696424899
GloVe Epoch: 5, Loss: 300.17471856553976
GloVe Epoch: 6, Loss: 291.6851563110558
GloVe Epoch: 7, Loss: 283.5580957107565
GloVe Epoch: 8, Loss: 275.77687243879853
GloVe Epoch: 9, Loss: 268.3256059230215
GloVe Epoch: 10, Loss: 261.1891621222317

Experiment 1: OOV Behavior
GloVe:
teachable: No
unteacher: No
supercomputer: No
miscomputation: No
unnaturally: No
Word2Vec:
teachable: No
unteacher: No
supercomputer: No
miscomputation: No
unnaturally: No
FastText:
teachable: Yes
unteacher: Yes
supercomputer: Yes
miscomputation: Yes
unnaturally: Yes

Experiment 2: Rare Words
GloVe:
(unteachable, teacher): 0.1431520035026872
(recomputation, computation): -0.1306761537415205
(unnaturalness, natural): -0.09258228745312863
Word2Vec:
(unteachable, teacher): 0.4313947856426239
(recomputation, computation): 0.1272691786289215
(unnaturalness, na